In [3]:
import torch
import torch.nn as nn
import torchvision.datasets as dsets
import torchvision.transforms as transforms
import torch.nn.init

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(777)
if device == torch.device("cuda"):
    torch.cuda.manual_seed_all(777)
learning_rate = 0.001
training_epochs = 15
batch_size = 100

mnist_train = dsets.MNIST(root='MNIST_data/',
                          train=True,
                          transform=transforms.ToTensor(),
                          download=True)
mnist_test = dsets.MNIST(root='MNIST_data/',
                         train=False,
                         transform=transforms.ToTensor(),
                         download=True)
data_loader = torch.utils.data.DataLoader(dataset=mnist_train,
                                          batch_size=batch_size,
                                          shuffle=True,
                                          drop_last=True)

class CNN(torch.nn.Module):
    def __init__(self):
        super(CNN,self).__init__()
        dropout_prob = 0.3
        self.layer1 = torch.nn.Sequential(
            torch.nn.Conv2d(1,32,kernel_size=3,stride=1,padding=1),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(kernel_size = 2,stride = 2)
        )
        self.layer2 = torch.nn.Sequential(
            torch.nn.Conv2d(32,64,kernel_size=3,stride=1,padding=1),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(kernel_size = 2,stride = 2)
        )
        self.layer3 = torch.nn.Sequential(
            torch.nn.Conv2d(64,128,kernel_size=3,stride=1,padding=1),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(kernel_size = 2,stride = 2)
        )
        self.fc1 = torch.nn.Linear(3*3*128,625,bias=True)
        torch.nn.init.xavier_uniform_(self.fc1.weight)
        self.layer4 = torch.nn.Sequential(
            self.fc1,
            torch.nn.ReLU(),
            torch.nn.Dropout(p=dropout_prob)
        )
        self.fc2 = torch.nn.Linear(625,10,bias=True)
        torch.nn.init.xavier_uniform_(self.fc2.weight)
        
    def forward(self,x):
        out = self.layer1(x)
        out = self.layer2(out)
        out = self.layer3(out)
        out = out.view(out.size(0),-1)
        out = self.layer4(out)
        out = self.fc2(out)
        return out

criterion = torch.nn.CrossEntropyLoss().to(device)
model = CNN().to(device)
optimizer = torch.optim.Adam(model.parameters(),lr=learning_rate)
total_batch = len(data_loader)
for epoch in range(training_epochs):
    avg_cost = 0
    for X,Y in data_loader:
        X = X.to(device)
        Y = Y.to(device)
        
        optimizer.zero_grad()
        hypothesis = model(X)
        cost = criterion(hypothesis,Y)
        cost.backward()
        optimizer.step()
        
        avg_cost += cost / total_batch
    print('[Epoch: {:>4}] cost = {:>.9}'.format(epoch+1, avg_cost))
    
with torch.no_grad():
    X_test = mnist_test.test_data.view(len(mnist_test),1,28,28).float().to(device)
    Y_test = mnist_test.test_labels.to(device)
    
    prediction = model(X_test)
    correct_prediction = torch.argmax(prediction,1) == Y_test
    accuracy = correct_prediction.float().mean()
    print('Accuracy:', accuracy.item())

[Epoch:    1] cost = 0.181045607
[Epoch:    2] cost = 0.0486124009
[Epoch:    3] cost = 0.0342535339
[Epoch:    4] cost = 0.0254438575
[Epoch:    5] cost = 0.0211183019
[Epoch:    6] cost = 0.0177106485
[Epoch:    7] cost = 0.0140847862
[Epoch:    8] cost = 0.0138729336
[Epoch:    9] cost = 0.0104293106
[Epoch:   10] cost = 0.0103954664
[Epoch:   11] cost = 0.00949621014
[Epoch:   12] cost = 0.00810521003
[Epoch:   13] cost = 0.00746852346
[Epoch:   14] cost = 0.00688993093
[Epoch:   15] cost = 0.00697807223
Accuracy: 0.990399956703186


d:\deeplearning\.venv\Lib\site-packages\torchvision\datasets\mnist.py:81: UserWarning: test_data has been renamed data
  warnings.warn("test_data has been renamed data")
d:\deeplearning\.venv\Lib\site-packages\torchvision\datasets\mnist.py:71: UserWarning: test_labels has been renamed targets
  warnings.warn("test_labels has been renamed targets")


In [6]:
import torch
import torch.nn as nn
import torchvision
import torchvision.datasets as dsets
import visdom
import torchvision.transforms as transforms
vis = visdom.Visdom()

MNIST = dsets.MNIST(root='MNIST_data/',
                         train=True,
                         transform=transforms.ToTensor(),
                         download=True)
cifar10 = dsets.CIFAR10(root='CIFAR10_data/',
                         train=True,
                         transform=transforms.ToTensor(),
                         download=True)
data = MNIST.__getitem__(0)
print(data[0].shape)
vis.image(data[0],env = "main")

data_loader = torch.utils.data.DataLoader(dataset=MNIST,
                                          batch_size=32,
                                          shuffle=False)
for num,value in enumerate(data_loader):
    value = value[0]
    print(value.shape)
    vis.images(value)
    break

vis.close(env="main")

Y_data = torch.randn(5)
plt = vis.line(Y = Y_data)

X_data = torch.Tensor([1,2,3,4,5])
plt = vis.line(Y = Y_data, X = X_data)

Y_append = torch.randn(1)
X_append = torch.Tensor([6])
vis.line(Y=Y_append, X=X_append, win=plt, update='append')
num = torch.Tensor(list(range(0,10)))
num = num.view(-1,1)
num = torch.cat((num,num),dim = 1)

plt = vis.line(Y = torch.randn(10,2), X = num)


Setting up a new session...


torch.Size([1, 28, 28])
torch.Size([32, 1, 28, 28])


MNIST-CNN with Visdom

In [ ]:
import torch
import torch.nn as nn
import torchvision.datasets as dsets
import torchvision.transforms as transforms
import torch.nn.init
import visdom

visdom = visdom.Visdom() # visdom的作用是
vis.close(env="main")

def loss_tracker(loss_plot,loss_value,num):
    vis.line(X = num,Y = loss_value,win = loss_plot,update = 'append')#win代表要更新的图
    
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(777)
if device == torch.device("cuda"):
    torch.cuda.manual_seed_all(777)
    
learning_rate = 0.001
training_epochs = 15
batch_size = 32
mnist_train = dsets.MNIST(root='MNIST_data/',
                          train=True,
                          transform=transforms.ToTensor(),
                          download=True)
mnist_test = dsets.MNIST(root='MNIST_data/',
                            train=False,
                            transform=transforms.ToTensor(),
                            download=True)
data_loader = torch.utils.data.DataLoader(dataset=mnist_train,
                                          batch_size=batch_size,
                                            shuffle=True,
                                            drop_last=True)
class CNN(torch.nn.Module):
    def __init__(self):
        super(CNN,self).__init__()
        self.layer1 = torch.nn.Sequential(
            torch.nn.Conv2d(1,32,kernel_size=3,stride=1,padding=1),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(2)
        )
        self.layer2 = torch.nn.Sequential(
            torch.nn.Conv2d(32,64,kernel_size=3,stride=1,padding=1),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(2)
        )
        self.layer3 = torch.nn.Sequential(
            torch.nn.Conv2d(64,128,kernel_size=3,stride=1,padding=1),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(2)
        )
        self.fc1 = torch.nn.Linear(3*3*128,625,bias=True)
        torch.nn.init.xavier_uniform_(self.fc1.weight)
        self.relu = torch.nn.ReLU()
        self.fc2 = torch.nn.Linear(625,10,bias=True)
        torch.nn.init.xavier_uniform_(self.fc2.weight)
    def forward(self,x):
        out = self.layer1(x)
        out = self.layer2(out)
        out = self.layer3(out)
        out = out.view(out.size(0),-1)
        out = self.fc1(out)
        out = self.relu(out)
        out = self.fc2(out)
        return out
    
criterion = torch.nn.CrossEntropyLoss().to(device)
model = CNN().to(device)
optimizer = torch.optim.Adam(model.parameters(),lr=learning_rate)
value = (torch.Tensor(1,1,28,28)).to(device)
print(model(value).shape)

loss_plt = vis.line(Y = torch.Tensor(1).zero_(),opts=dict(title="loss_tracker",legend=['loss'],showlegend=True))

for epoch in range(training_epochs):
    avg_cost = 0
    for X,Y in data_loader:
        X = X.to(device)
        Y = Y.to(device)
        hypothesis = model(X)
        optimizer.zero_grad()
        cost = criterion(hypothesis,Y)
        cost.backward()
        optimizer.step()
        avg_cost += cost / len(data_loader)
    print('[Epoch: {:>4}] cost = {:>.9}'.format(epoch+1, avg_cost))
    loss_tracker(loss_plt,torch.Tensor([avg_cost]).to(device),torch.Tensor([epoch+1]).to(device))
with torch.no_grad():
    X_test = mnist_test.test_data.view(len(mnist_test),1,28,28).float().to(device)
    Y_test = mnist_test.test_labels.to(device)
    
    prediction = model(X_test)
    correct_prediction = torch.argmax(prediction,1) == Y_test
    accuracy = correct_prediction.float().mean()
    print('Accuracy:', accuracy.item())
    

Setting up a new session...


torch.Size([1, 10])
[Epoch:    1] cost = 0.128622815
[Epoch:    2] cost = 0.0423202589
[Epoch:    3] cost = 0.0303848069
[Epoch:    4] cost = 0.0230704565
[Epoch:    5] cost = 0.0179240722
[Epoch:    6] cost = 0.0152444774
[Epoch:    7] cost = 0.0133539932
[Epoch:    8] cost = 0.0110974815
[Epoch:    9] cost = 0.0106023494
[Epoch:   10] cost = 0.00811727066
[Epoch:   11] cost = 0.00737234671
[Epoch:   12] cost = 0.00903336704
[Epoch:   13] cost = 0.00728032365
[Epoch:   14] cost = 0.00737528363
[Epoch:   15] cost = 0.00666171126
Accuracy: 0.9851999878883362


d:\deeplearning\.venv\Lib\site-packages\torchvision\datasets\mnist.py:81: UserWarning: test_data has been renamed data
  warnings.warn("test_data has been renamed data")
d:\deeplearning\.venv\Lib\site-packages\torchvision\datasets\mnist.py:71: UserWarning: test_labels has been renamed targets
  warnings.warn("test_labels has been renamed targets")


ImageFolder

In [2]:
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader
from matplotlib.pyplot import imshow
%matplotlib inline

trans = transforms.Compose([transforms.Resize((64,128))])
train_data = torchvision.datasets.ImageFolder(root='custom_data/origin_data', transform=trans)

for num,value in enumerate(train_data):
    data,label = value
    print(num,data,label)
    if(label == 0):
        data.save('custom_data/train_data/gray/%d_%d.jpeg'%(num, label))
    else:
        data.save('custom_data/train_data/red/%d_%d.jpeg'%(num, label))

0 <PIL.Image.Image image mode=RGB size=128x64 at 0x274B89EF9D0> 0
1 <PIL.Image.Image image mode=RGB size=128x64 at 0x274F21A8050> 0
2 <PIL.Image.Image image mode=RGB size=128x64 at 0x274F20DF0B0> 0
3 <PIL.Image.Image image mode=RGB size=128x64 at 0x274F216CF30> 0
4 <PIL.Image.Image image mode=RGB size=128x64 at 0x274F20E6350> 0
5 <PIL.Image.Image image mode=RGB size=128x64 at 0x274F21A2210> 0
6 <PIL.Image.Image image mode=RGB size=128x64 at 0x274F20C5FD0> 0
7 <PIL.Image.Image image mode=RGB size=128x64 at 0x274F2033110> 0
8 <PIL.Image.Image image mode=RGB size=128x64 at 0x274F1FE3A10> 0
9 <PIL.Image.Image image mode=RGB size=128x64 at 0x274F21A60D0> 0
10 <PIL.Image.Image image mode=RGB size=128x64 at 0x274F21A6170> 0
11 <PIL.Image.Image image mode=RGB size=128x64 at 0x274F21A56D0> 0
12 <PIL.Image.Image image mode=RGB size=128x64 at 0x274F21A60D0> 0
13 <PIL.Image.Image image mode=RGB size=128x64 at 0x274F21A6170> 0
14 <PIL.Image.Image image mode=RGB size=128x64 at 0x274F21A56D0> 0
15 <P

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as functional    
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.datasets as dsets

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(777)
if device == torch.device("cuda"):
    torch.cuda.manual_seed_all(777)
    
trans = transforms.Compose([transforms.Resize((64,128)), transforms.ToTensor()])
train_data = torchvision.datasets.ImageFolder(root='custom_data/train_data', transform=trans)
data_loader = DataLoader(dataset=train_data, batch_size=32, shuffle=True, drop_last=True,num_workers=2) 

class CNN(torch.nn.Module):
    def __init__(self):
        super(CNN,self).__init__()
        self.layer1 = torch.nn.Sequential(
            torch.nn.Conv2d(3,6,5),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(2,2)
        )
        self.layer2 = torch.nn.Sequential(
            torch.nn.Conv2d(6,16,5),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(2,2)
        )
        self.layer3 = torch.nn.Sequential(
            torch.nn.Linear(16*13*29,120),
            torch.nn.ReLU(),
            torch.nn.Linear(120,2),
        )
    def forward(self,x):
        out = self.layer1(x)
        out = self.layer2(out)
        out = out.view(out.size(0),-1)
        out = self.layer3(out)
        return out
    
model = CNN().to(device)
test_input = (torch.Tensor(3,3,64,128)).to(device)
test_output = model(test_input) 
criterion = torch.nn.CrossEntropyLoss().to(device) 
optimizer = torch.optim.Adam(model.parameters(),lr=0.001)
total_batch = len(data_loader)
epochs = 7
for epoch in range(epochs):
    avg_cost = 0
    for X,Y in data_loader:
        X = X.to(device)
        Y = Y.to(device)
        
        optimizer.zero_grad()
        hypothesis = model(X)
        cost = criterion(hypothesis,Y)
        cost.backward()
        optimizer.step()
        
        avg_cost += cost / total_batch
    print('[Epoch: {:>4}] cost = {:>.9}'.format(epoch+1, avg_cost))

import os # 1. 导入 os 模块

# 2. 创建模型保存目录
model_dir = './model'
if not os.path.exists(model_dir):
    os.makedirs(model_dir)
torch.save(model.state_dict(), 'custom_cnn_model.pth')
torch.save(model.state_dict(), "./model/model.pth")
new_net = CNN().to(device)
new_net.load_state_dict(torch.load('./model/model.pth'))
print(model.layer1[0])
print(new_net.layer1[0])

print(model.layer1[0].weight[0][0][0])
print(new_net.layer1[0].weight[0][0][0])

model.layer1[0].weight[0] == new_net.layer1[0].weight[0]

print("Model saved and loaded successfully.")
trans = torchvision.transforms.Compose([torchvision.transforms.Resize((64,128)), torchvision.transforms.ToTensor()])
test_data = torchvision.datasets.ImageFolder(root='custom_data/test_data', transform=trans)
test_loader = DataLoader(dataset=test_data, batch_size=1, shuffle=False)
with torch.no_grad():
    for num, value in enumerate(test_loader):
        X, Y = value
        X = X.to(device)
        Y = Y.to(device)
        
        prediction = new_net(X)
        predicted_label = torch.argmax(prediction,1)
        print("True Label: {}, Predicted Label: {}".format(Y.item(), predicted_label.item()))
    
       

[Epoch:    1] cost = 0.51049006
[Epoch:    2] cost = 0.0565474294
[Epoch:    3] cost = 0.00212598569
[Epoch:    4] cost = 0.000420061639
[Epoch:    5] cost = 0.000143347555
[Epoch:    6] cost = 0.000115777177
[Epoch:    7] cost = 7.87377139e-05
Conv2d(3, 6, kernel_size=(5, 5), stride=(1, 1))
Conv2d(3, 6, kernel_size=(5, 5), stride=(1, 1))
tensor([-0.0945, -0.0002, -0.0208, -0.0252,  0.0889], device='cuda:0',
       grad_fn=<SelectBackward0>)
tensor([-0.0945, -0.0002, -0.0208, -0.0252,  0.0889], device='cuda:0',
       grad_fn=<SelectBackward0>)
Model saved and loaded successfully.
True Label: 0, Predicted Label: 0
True Label: 0, Predicted Label: 0
True Label: 0, Predicted Label: 0
True Label: 0, Predicted Label: 0
True Label: 0, Predicted Label: 0
True Label: 0, Predicted Label: 0
True Label: 0, Predicted Label: 0
True Label: 0, Predicted Label: 0
True Label: 0, Predicted Label: 0
True Label: 0, Predicted Label: 0
True Label: 0, Predicted Label: 0
True Label: 0, Predicted Label: 0
True

VGG

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import visdom

vis = visdom.Visdom()
vis.close(env="main")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(777)
if device == torch.device("cuda"):
    torch.cuda.manual_seed_all(777)

def loss_tracker(loss_plot,loss_value,num):
    vis.line(X = num,Y = loss_value,win = loss_plot,update = 'append')#win代表要更新的图
    
transform = transforms.Compose([transforms.ToTensor(),
                                transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

trainset = torchvision.datasets.CIFAR10(root='./CIFAR10_data', train=True,
                                        download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=32,
                                          shuffle=True, num_workers=2)
testset = torchvision.datasets.CIFAR10(root='./CIFAR10_data', train=False,
                                       download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=32,
                                         shuffle=False, num_workers=2)
classes = ('plane', 'car', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck')

import matplotlib.pyplot as plt
import numpy as np
%matplotlib inline
def imshow(img):
    img = img/2 - 0.5
    img = img.numpy()
    plt.imshow(np.transpose(img, (1, 2, 0)))
    plt.show()
    
dataiter = iter(trainloader)
images, labels = next(dataiter)
vis.images(images/2 - 0.5)
print(' '.join('%5s' % classes[labels[j]] for j in range(4)))

import torchvision.models.vgg as vgg
cfg = [32,32,"M",64,64,128,128,128,"M",256,256,256,512,512,512,"M"]
class VGG(nn.Module):
    def __init__(self,feature,num_classes,init_weights=True):
        super(VGG,self).__init__()
        self.features = feature
        self.classifier = nn.Sequential(
            nn.Linear(512*4*4,4096),
            nn.ReLU(True),
            nn.Dropout(),
            nn.Linear(4096,4096),
            nn.ReLU(True),
            nn.Dropout(),
            nn.Linear(4096,num_classes)
        )
    def forward(self,x):
        x = self.features(x)
        x = x.view(x.size(0),-1)
        x = self.classifier(x)
        return x
    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m,nn.Conv2d):
                nn.init.kaiming_normal_(m.weight,mode='fan_out',nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias,0)
            elif isinstance(m,nn.BatchNorm2d):
                nn.init.constant_(m.weight,1)
                nn.init.constant_(m.bias,0)
            elif isinstance(m,nn.Linear):
                nn.init.normal_(m.weight,0,0.01)
                nn.init.constant_(m.bias,0)

vgg16 = VGG(vgg.make_layers(cfg),10).to(device)
criterion = nn.CrossEntropyLoss().to(device)
optimizer = optim.SGD(vgg16.parameters(),lr=0.01,momentum=0.9)
criterion = torch.nn.CrossEntropyLoss().to(device)
lr_scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.9)
loss_plt = vis.line(Y = torch.Tensor(1).zero_(),opts=dict(title="VGG_loss_tracker",legend=['loss'],showlegend=True))
for epoch in range(50):
    avg_cost = 0
    for X,Y in trainloader:
        X = X.to(device)
        Y = Y.to(device)
        
        optimizer.zero_grad()
        hypothesis = vgg16(X)
        cost = criterion(hypothesis,Y)
        cost.backward()
        optimizer.step()
        
        avg_cost += cost / len(trainloader)
    print('[Epoch: {:>4}] cost = {:>.9}'.format(epoch+1, avg_cost))
    loss_tracker(loss_plt,torch.Tensor([avg_cost]).to(device),torch.Tensor([epoch+1]).to(device))
    lr_scheduler.step()
    
dataiter = iter(testloader)
images, labels = next(dataiter)
imshow(torchvision.utils.make_grid(images))
print('GroundTruth: ', ' '.join('%5s' % classes[labels[j]] for j in range(4)))
outputs = vgg16(images.to(device))
_, predicted = torch.max(outputs, 1)
print('Predicted: ', ' '.join('%5s' % classes[predicted[j]] for j in range(4)))

with torch.no_grad():
    correct = 0
    total = 0
    for X,Y in testloader:
        X = X.to(device)
        Y = Y.to(device)
        
        prediction = vgg16(X)
        predicted_label = torch.argmax(prediction,1)
        total += Y.size(0)
        correct += (predicted_label == Y).sum().item()
    print('Accuracy:', correct / total)

Setting up a new session...


truck   car   car   cat
[Epoch:    1] cost = 2.30361485
[Epoch:    2] cost = 2.30368209
[Epoch:    3] cost = 2.30366397
[Epoch:    4] cost = 2.30348897
[Epoch:    5] cost = 2.30372119
[Epoch:    6] cost = 2.3035512
[Epoch:    7] cost = 2.30355573
[Epoch:    8] cost = 2.30341697
[Epoch:    9] cost = 2.30332541
[Epoch:   10] cost = 2.30337358


KeyboardInterrupt: 

[WinError 10054] 远程主机强迫关闭了一个现有的连接。
Visdom.setup_socket.<locals>.on_close() takes 1 positional argument but 3 were given
[WinError 10061] 由于目标计算机积极拒绝，无法连接。
Visdom.setup_socket.<locals>.on_close() takes 1 positional argument but 3 were given
[WinError 10061] 由于目标计算机积极拒绝，无法连接。
Visdom.setup_socket.<locals>.on_close() takes 1 positional argument but 3 were given
[WinError 10061] 由于目标计算机积极拒绝，无法连接。
Visdom.setup_socket.<locals>.on_close() takes 1 positional argument but 3 were given
[WinError 10061] 由于目标计算机积极拒绝，无法连接。
Visdom.setup_socket.<locals>.on_close() takes 1 positional argument but 3 were given
[WinError 10061] 由于目标计算机积极拒绝，无法连接。
Visdom.setup_socket.<locals>.on_close() takes 1 positional argument but 3 were given
[WinError 10061] 由于目标计算机积极拒绝，无法连接。
Visdom.setup_socket.<locals>.on_close() takes 1 positional argument but 3 were given
[WinError 10061] 由于目标计算机积极拒绝，无法连接。
Visdom.setup_socket.<locals>.on_close() takes 1 positional argument but 3 were given
[WinError 10061] 由于目标计算机积极拒绝，无法连接。
Visdo

Residual Network

In [ ]:
import torch
import torchvision
import torch.nn as nn
import torchvision.transforms as transforms
import torch.nn.functional as F
import torch.optim as optim
import visdom

vis = visdom.Visdom()
vis.close(env="main")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(777)
if device == torch.device("cuda"):
    torch.cuda.manual_seed_all(777)
    
def value_tracker(plot,value_value,num):
    vis.line(X = num,Y = value_value,win = plot,update = 'append')#win代表要更新的图

train_set = torchvision.datasets.CIFAR10(root='./CIFAR10_data', train=True,
                                        download=True, transform=transforms.ToTensor())

transform = transforms.Compose([transforms.RandomCrop(32, padding=4),
                                transforms.ToTensor(),
                                transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))]) 
trainset = torchvision.datasets.CIFAR10(root='./CIFAR10_data', train=True,
                                        download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=32,
                                          shuffle=True, num_workers=0)
testset = torchvision.datasets.CIFAR10(root='./CIFAR10_data', train=False,
                                       download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=32,
                                         shuffle=False, num_workers=0)

classes = ('plane', 'car', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck')

import torchvision.models.resnet as resnet
con1x1 = resnet.conv1x1
Bottleneck = resnet.Bottleneck
Basicneck = resnet.BasicBlock
class ResNet(nn.Module):
    def __init__(self,block,layers,num_classes=10,zero_init_residual=False):  
        super(ResNet,self).__init__()
        self.inplanes = 16
        self.con1 = nn.Conv2d(3,16,kernel_size=3,stride=1,padding=1,bias=False)
        self.bn1 = nn.BatchNorm2d(16)
        self.relu = nn.ReLU(inplace=True)
        self.layer1 = self._make_layer(block,16,layers[0],stride=1)
        self.layer2 = self._make_layer(block,32,layers[1],stride=1)
        self.layer3 = self._make_layer(block,64,layers[2],stride=2)
        self.layer4 = self._make_layer(block,128,layers[3],stride=2)
        self.avgpool = nn.AdaptiveAvgPool2d((1,1))
        self.fc = nn.Linear(128*block.expansion,num_classes)
        
        for m in self.modules():
            if isinstance(m,nn.Conv2d):
                nn.init.kaiming_normal_(m.weight,mode='fan_out',nonlinearity='relu')
            elif isinstance(m,nn.BatchNorm2d):
                nn.init.constant_(m.weight,1)         
                nn.init.constant_(m.bias,0)

        if zero_init_residual:
            for m in self.modules():
                if isinstance(m,Bottleneck):
                    nn.init.constant_(m.bn3.weight,0)
                elif isinstance(m,Basicneck):
                    nn.init.constant_(m.bn2.weight,0)
                    
    def _make_layer(self,block,planes,blocks,stride):
        downsample = None
        if stride != 1 or self.inplanes !=planes * block.expansion:
            downsample = nn.Sequential(
                con1x1(self.inplanes, planes * block.expansion, stride),
                nn.BatchNorm2d(planes * block.expansion),
            )
                
        layers = []
        layers.append(block(self.inplanes,planes,stride,downsample))
        self.inplanes = planes * block.expansion
        for _ in range(1,blocks):
            layers.append(block(self.inplanes,planes))          
        return nn.Sequential(*layers)
    def forward(self,x):
        x = self.con1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = x.view(x.size(0),-1)
        x = self.fc(x)
        return x

resnet50 = ResNet(Bottleneck,[3,4,6,3]).to(device)
optimizer = optim.SGD(resnet50.parameters(),lr=0.1,momentum=0.9,weight_decay=5e-4)
lr_scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
loss_plt = vis.line(Y = torch.Tensor(1).zero_(),opts=dict(title="ResNet_loss_tracker",legend=['loss'],showlegend=True))
acc_plt = vis.line(Y = torch.Tensor(1).zero_(),opts=dict(title="ResNet_acc_tracker",legend=['acc'],showlegend=True))
criterion = nn.CrossEntropyLoss().to(device)
def acc_check(net, test_set, epoch, save=1):
    correct = 0
    total = 0
    with torch.no_grad():
        for data in test_set:
            images, labels = data
            images = images.to(device)
            labels = labels.to(device)
            outputs = net(images)

            _, predicted = torch.max(outputs.data, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    acc = (100 * correct / total)
    print('Accuracy of the network on the 10000 test images: %d %%' % acc)
    if save:
        torch.save(net.state_dict(), "./model/model_epoch_{}_acc_{}.pth".format(epoch, int(acc)))
    return acc

for epoch in range(50):
    running_loss = 0.0
    for i,data in enumerate(trainloader,0):
        inputs,labels = data
        inputs = inputs.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = resnet50(inputs)
        loss = criterion(outputs,labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        if i % 30 == 29:    # print every 30 mini-batches
            value_tracker(loss_plt, torch.Tensor([running_loss/30]), torch.Tensor([i + epoch*len(trainloader) ]))
            print('[%d, %5d] loss: %.3f' %
                  (epoch + 1, i + 1, running_loss / 30))
            running_loss = 0.0
    
    #Check Accuracy
    acc = acc_check(resnet50, testloader, epoch, save=1)
    value_tracker(acc_plt, torch.Tensor([acc]), torch.Tensor([epoch]))

correct = 0
total = 0

with torch.no_grad():
    for data in testloader:
        images, labels = data
        images = images.to(device)
        labels = labels.to(device)
        outputs = resnet50(images)
        
        _, predicted = torch.max(outputs.data, 1)
        
        total += labels.size(0)
        
        correct += (predicted == labels).sum().item()

print('Accuracy of the network on the 10000 test images: %d %%' % (
    100 * correct / total))
    
    

Setting up a new session...


[1,    30] loss: 5.535
[1,    60] loss: 2.335
[1,    90] loss: 2.317
[1,   120] loss: 2.310
[1,   150] loss: 2.296
[1,   180] loss: 2.264
[1,   210] loss: 2.244
[1,   240] loss: 2.246
[1,   270] loss: 2.218
[1,   300] loss: 2.172
[1,   330] loss: 2.179
[1,   360] loss: 2.142
[1,   390] loss: 2.153
[1,   420] loss: 2.132
[1,   450] loss: 2.102
[1,   480] loss: 2.049
[1,   510] loss: 2.095
[1,   540] loss: 2.087
[1,   570] loss: 2.030
[1,   600] loss: 2.060
[1,   630] loss: 1.974
[1,   660] loss: 1.995
[1,   690] loss: 2.017
[1,   720] loss: 1.970
[1,   750] loss: 1.964
[1,   780] loss: 1.941
[1,   810] loss: 1.981
[1,   840] loss: 1.968
[1,   870] loss: 1.932
[1,   900] loss: 1.979
[1,   930] loss: 1.933
[1,   960] loss: 1.932
[1,   990] loss: 1.916
[1,  1020] loss: 1.867
[1,  1050] loss: 1.849
[1,  1080] loss: 1.938
[1,  1110] loss: 1.894
[1,  1140] loss: 1.857
[1,  1170] loss: 1.902
[1,  1200] loss: 1.825
[1,  1230] loss: 1.877
[1,  1260] loss: 1.780
[1,  1290] loss: 1.833
[1,  1320] 

KeyboardInterrupt: 

[WinError 10054] 远程主机强迫关闭了一个现有的连接。
Visdom.setup_socket.<locals>.on_close() takes 1 positional argument but 3 were given
[WinError 10061] 由于目标计算机积极拒绝，无法连接。
Visdom.setup_socket.<locals>.on_close() takes 1 positional argument but 3 were given
[WinError 10061] 由于目标计算机积极拒绝，无法连接。
Visdom.setup_socket.<locals>.on_close() takes 1 positional argument but 3 were given
[WinError 10061] 由于目标计算机积极拒绝，无法连接。
Visdom.setup_socket.<locals>.on_close() takes 1 positional argument but 3 were given
[WinError 10061] 由于目标计算机积极拒绝，无法连接。
Visdom.setup_socket.<locals>.on_close() takes 1 positional argument but 3 were given
[WinError 10061] 由于目标计算机积极拒绝，无法连接。
Visdom.setup_socket.<locals>.on_close() takes 1 positional argument but 3 were given
[WinError 10061] 由于目标计算机积极拒绝，无法连接。
Visdom.setup_socket.<locals>.on_close() takes 1 positional argument but 3 were given
[WinError 10061] 由于目标计算机积极拒绝，无法连接。
Visdom.setup_socket.<locals>.on_close() takes 1 positional argument but 3 were given
[WinError 10061] 由于目标计算机积极拒绝，无法连接。
Visdo